# Desafio Técnico Artifact AI Engineer 2026

In [74]:
import warnings
warnings.filterwarnings('ignore')

from IPython.display import display, Markdown

In [75]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()

# Configuração do LLM apontando para o LM Studio
# llm = ChatOpenAI(
#     base_url="http://localhost:1234/v1",
#     api_key="lm-studio",
#     model="qwen/qwen3.5-9b",
#     temperature=0
# )

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

## Agente SQL

In [76]:
import pandas as pd
from sqlalchemy import create_engine

# 1. Criação do Banco de Dados SQLite local
engine = create_engine("sqlite:///loja_musica.db")

# Dicionário mapeando o nome do arquivo para o nome da tabela no SQL
arquivos_csv = {
    "categories": "data/desafio_tecnico_ai_eng - categories.csv",
    "customers": "data/desafio_tecnico_ai_eng - customers.csv",
    "order_items": "data/desafio_tecnico_ai_eng - order_items.csv",
    "orders": "data/desafio_tecnico_ai_eng - orders.csv",
    "products": "data/desafio_tecnico_ai_eng - products.csv",
    "promotions": "data/desafio_tecnico_ai_eng - promotions.csv"
}

# Leitura dos CSVs e inserção no banco de dados
# for nome_tabela, caminho_arquivo in arquivos_csv.items():
#     df = pd.read_csv(caminho_arquivo)
#     # Salva o DataFrame como uma tabela SQL. replace garante que reescreva se rodar de novo.
#     df.to_sql(nome_tabela, engine, index=False, if_exists="replace")

# print("Banco de dados loja_musica.db criado com sucesso!")


In [ ]:

from langchain_community.utilities import SQLDatabase
from langchain_core.prompts import PromptTemplate
from langchain_community.agent_toolkits import create_sql_agent

# Conexão do LangChain ao banco de dados
db = SQLDatabase(engine=engine)

sql_system_rules_prompt = PromptTemplate.from_template("""Você é um assistente de IA especializado em consultas SQL para uma loja de instrumentos musicais.

Responda sempre em Português Brasileiro.

Seja específico e detalhado em suas consultas e respostas, fornecendo informações precisas do banco de dados.

Para consultas de produtos, leve em consideração o preço, a categoria e a disponibilidade em estoque.

Responda a pergunta com detalhes e informações do banco de dados:

{question}
""")

sql_llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

# Criação do Agente SQL
sql_agent = create_sql_agent(
    llm,
    db=db,
    agent_type="openai-tools",
    # verbose=True # Mantém o log ativo ver a query SQL gerada no terminal,
)

In [ ]:
sql_chain = sql_system_rules_prompt | sql_agent

resposta = sql_chain.invoke({"question": "Qual foi o cliente que fez o pedido mais caro?"})

display(Markdown(resposta["output"]))

O cliente que fez o pedido mais caro foi **Pedro Henrique Oliveira**, com um total de **R$ 11.499,00**.

In [104]:
sql_chain = sql_system_rules_prompt | sql_agent

resposta = sql_chain.invoke({"question": "Quais são as três categorias com mais produtos cadastrados?"})

display(Markdown(resposta["output"]))

As três categorias com mais produtos cadastrados são:

1. **Violões** - 35 produtos
2. **Ukuleles** - 15 produtos
3. **Guitarras** - 5 produtos

Essas categorias refletem a variedade de instrumentos disponíveis na loja.

In [117]:
sql_chain = sql_system_rules_prompt | sql_agent

resposta = sql_chain.invoke({"question": "Qual são os três violões mais baratos?"} )

display(Markdown(resposta["output"]))

Os três violões mais baratos disponíveis em estoque são:

1. **Tagima Memphis AC-39 Nylon Natural** - R$ 429,90
2. **Yamaha C40 Nylon Natural** - R$ 599,90
3. **Yamaha C70 Nylon Natural** - R$ 849,00

Esses violões estão ativos e prontos para venda.

### Com memória

In [ ]:
# Dicionário para armazenar o estado (resumo) de cada sessão na memória RAM
resumos_sessao = {}

def consultar_com_resumo(pergunta_usuario: str, session_id: str) -> str:
    # 1. Recupera o resumo atual do usuário
    resumo_atual = resumos_sessao.get(session_id, "")
    
    # 2. Injeta o contexto no prompt do agente
    if resumo_atual:
        input_agente = f"Contexto histórico:\n{resumo_atual}\n\nNova pergunta: {pergunta_usuario}"
    else:
        input_agente = pergunta_usuario
        
    # 3. Invoca o agent_executor existente
    resposta = sql_agent.invoke({"input": input_agente})
    
    # 4. Atualiza o resumo em background usando o seu LLM local
    prompt_resumo = f"""
    Resumo anterior: {resumo_atual}
    Novo input do usuário: {pergunta_usuario}
    Nova resposta da IA: {resposta['output']}
    
    Sintetize o contexto atualizado desta conversa de forma concisa para servir de memória futura.
    """
    
    # Chama o qwen3.5 puramente para resumir
    novo_resumo = llm.invoke(prompt_resumo).content
    resumos_sessao[session_id] = novo_resumo
    
    return resposta["output"]

# Testando a interação
display(Markdown(consultar_com_resumo("Qual o cliente que fez o pedido mais recente?", "session_01")))
display(Markdown(consultar_com_resumo("Quais produtos ele comprou?", "session_01")))

O cliente que fez o pedido mais recente é Camila Duarte Nascimento, com o pedido datado de 22 de março de 2026.

Camila Duarte Nascimento comprou 1 unidade do produto "Tagima Memphis AC-39 Nylon Natural".

In [135]:
resposta = sql_chain.invoke({"question": "Quais as três categorias de produtos mais vendidas?"})
display(Markdown(resposta["output"]))

As três categorias de produtos mais vendidas na loja de instrumentos musicais são:

1. **Violões** - 13 unidades vendidas
2. **Ukuleles** - 9 unidades vendidas
3. **Baixos** - 1 unidade vendida

Essas informações refletem as categorias com maior volume de vendas até o momento.

## PDF RAG

In [81]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# 1. Configura o modelo de embedding MULTILÍNGUE
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

diretorio_chroma = "./chroma_db_pt"

# 2. Lógica para não reprocessar o PDF à toa
if not os.path.exists(diretorio_chroma):
    print("Processando PDF e criando banco vetorial pela primeira vez...")
    loader = PyPDFLoader("data/políticas_da_loja.pdf")
    documentos = loader.load()
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
    chunks = text_splitter.split_documents(documentos)
    
    vector_store = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory=diretorio_chroma
    )
else:
    print("Carregando banco vetorial existente do disco...")
    vector_store = Chroma(
        persist_directory=diretorio_chroma, 
        embedding_function=embeddings
    )

retriever = vector_store.as_retriever(search_kwargs={"k": 4})

# 3. Configuração do LLM
internal_llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

# 4. Cadeia RAG
template = """Use os trechos de contexto abaixo para responder à pergunta, se possível detalhadamente. 
Se a resposta não estiver no contexto, diga que não sabe. Não invente informações.

Contexto:
{context}

Pergunta: {question}

Resposta concisa:"""
prompt = PromptTemplate.from_template(template)

rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | internal_llm
    | StrOutputParser()
)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2371.52it/s]


Carregando banco vetorial existente do disco...


In [82]:
resposta = rag_chain.invoke("Qual é a política de devolução da loja?")
display(Markdown(resposta))

A política de devolução da loja permite que o cliente solicite a devolução em até 7 dias corridos após o recebimento do produto, sem necessidade de justificativa. O produto deve estar em sua embalagem original, sem sinais de uso, e com todos os acessórios e manuais. O reembolso será realizado na mesma forma de pagamento original em até 10 dias úteis, e o frete de devolução é por conta da loja em caso de arrependimento.

In [83]:
resposta = rag_chain.invoke("Qual é o horário de funcionamento da loja?")
display(Markdown(resposta))

O horário de funcionamento da loja é de segunda a sexta-feira, das 09:00 às 18:00, e aos sábados, das 09:00 às 13:00. A loja está fechada aos domingos e feriados. Em datas especiais, como Black Friday, Aniversário da Loja e Natal, o horário pode ser estendido.

In [84]:
resposta = rag_chain.invoke("Qual o prazo de entrega para envios fora da região metropolitana?")
display(Markdown(resposta))

O prazo de entrega para envios fora da região metropolitana de Campo Grande é de 3 a 8 dias úteis, utilizando os serviços da transportadora Jadlog.

## Agente de roteamento

In [90]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

# 1. Prompt de Classificação
prompt_classificacao = PromptTemplate.from_template(
    """Você é um roteador inteligente para uma loja de instrumentos musicais.
    Analise a pergunta e decida qual sistema deve responder.
    
    Sistemas:
    - "SQL": Para perguntas sobre vendas, clientes, pedidos, catálogo de produtos e promoções.
    - "PDF": Para perguntas sobre regras da loja, políticas de devolução, trocas e garantias.
    
    Pergunta: {question}
    
    Responda APENAS com a palavra SQL ou PDF:"""
)

cadeia_classificacao = prompt_classificacao | llm | StrOutputParser()

def delegar_tarefa(info):
    classificacao = info["classificacao"].strip().upper()
    pergunta = info["question"]
    
    print(f"\n[Agente Mestre delegou para: {classificacao}]")
    
    if "SQL" in classificacao:
        # Chama o agente do banco de dados
        resposta_sql = sql_chain.invoke(pergunta)
        return resposta_sql["output"]
    else:
        # Chama a chain do ChromaDB
        return rag_chain.invoke(pergunta)

# 3. Construção do Agente Mestre
agente_mestre = (
    {"classificacao": cadeia_classificacao, "question": RunnablePassthrough()}
    | RunnableLambda(delegar_tarefa)
)

## Agente Chatbot

In [113]:
user_question = "Qual é o violão mais barato?"


In [139]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

chatbot_prompt = PromptTemplate.from_template(
    """Você é um chatbot atendente simpático e extrovertido de uma loja de instrumentos musicais.
    Responda à pergunta de forma educada e amigável com os dados que foram obtidos pelo sistema.
    
    Pergunta: {question}

    Dados obtidos: {obtained_data}
    """
)

chatbot_chain = ( 
    { "question": RunnablePassthrough(), "obtained_data": agente_mestre } 
    | chatbot_prompt 
    | llm 
    | StrOutputParser() )

resposta_final = chatbot_chain.invoke(user_question)

print(resposta_final)


[Agente Mestre delegou para: PDF]
Olá! 😊 Que bom que você está aqui! Sobre a nossa política de devolução, é super tranquila! Você pode solicitar a devolução em até 7 dias corridos após receber o seu produto, e não precisa dar nenhuma justificativa. 

Só pedimos que o item esteja na embalagem original, sem sinais de uso, e que todos os acessórios e manuais estejam junto. O reembolso será feito na mesma forma de pagamento que você usou, em até 10 dias úteis. E se você estiver devolvendo por arrependimento, o frete fica por nossa conta! 

Se precisar de mais alguma coisa ou tiver outras dúvidas, é só me chamar! 🎶✨


In [ ]:
user_question = "Quanto custa o violão mais barato?"
resposta_final = chatbot_chain.invoke(user_question)
display(Markdown(resposta_final))


[Agente Mestre delegou para: SQL]


Olá! 🎶 Que alegria receber sua pergunta! O violão mais barato que temos por aqui é o modelo "Shelby SU-21S Soprano Sunburst", e ele custa apenas R$ 159,90! Uma ótima opção para quem está começando ou quer um instrumento leve e fácil de tocar. Se precisar de mais informações ou ajuda para escolher, estou aqui! 😊🎸

In [138]:
user_question = "Qual é a política de devolução da loja?"
resposta_final = chatbot_chain.invoke(user_question)
display(Markdown(resposta_final))


[Agente Mestre delegou para: PDF]


Olá! 😊 Que bom que você está aqui! Vou te explicar nossa política de devolução de forma bem simples e clara.

Você pode solicitar a devolução do seu produto em até 7 dias corridos após recebê-lo, e o melhor: não precisa dar justificativa! Apenas lembre-se de que o item deve estar na embalagem original, sem sinais de uso, e com todos os acessórios e manuais. 

O reembolso será feito na mesma forma de pagamento que você usou, em até 10 dias úteis. E se você estiver devolvendo por arrependimento, o frete de devolução é por nossa conta! 🎉

Agora, se o seu produto apresentar algum defeito de fabricação, você pode trocá-lo em até 30 dias após a compra. Depois desse prazo, é só acionar a garantia diretamente com o fabricante.

Se precisar de mais alguma coisa ou tiver outras dúvidas, estou aqui para ajudar! 🎶✨

In [30]:
display(Markdown(consultar_com_resumo("Quanto custa o violão mais barato?", "session_01")))
display(Markdown(consultar_com_resumo("Qual descrição dele?", "session_01")))

O violão mais barato disponível custa R$ 429,90.

O violão **Tagima Memphis AC-39 Nylon Natural** é descrito como:

"Violão clássico com cordas de nylon, perfeito para quem está começando. Corpo leve e confortável para tocar."

Se precisar de mais informações, estou à disposição!